# Brightfield organoid morphometrics

Pipeline over brightfield tiles in `data/`:

1. **Convert** — flatten every `data/Fileset_*/*.ome.tiff` into `data_tiff/` as plain `.tiff`.
2. **Segment** — micro-sam `vit_b_lm` + AIS decoder on the GPU, masks cached to `masks_raw/`.
3. **Curate** — one napari window; click bad objects to delete, then advance. Curated masks go to `masks_curated/`.
4. **Measure** — area, perimeter, hydraulic diameter (4A/P) per object → `organoid_morphometrics.xlsx`.

Filenames encode the experiment: `EDF_sigma-55_VB-D8FT-0007` → constructor `VB`, condition `D8FT`,
replicate `0007`. Conditions present: D2 (8), D5 (10), D8C (8), D8FT (8).

Per-object metrics (area, perimeter, hydraulic diameter) are directly comparable across all of these.
**Object counts per image are not** — D2 fields cover ~1.7× the area of D5/D8 fields, so any
density or count comparison needs normalising by field of view. `D5-0004` is the odd one out in
its own group and is worth confirming it belongs in the set.

### Curation controls

| key / action | effect |
|---|---|
| **left-click** an object | delete it (a click, not a drag — dragging still pans) |
| **`u`** | undo last deletion |
| **`n`** | save this image's mask and advance |
| **`x`** | skip this whole image (excluded from the Excel file, logged with a reason) |

Objects touching the image border are dropped automatically before you see them.

Each cell is restartable: conversion, segmentation and curation all skip work that's already on disk,
so you can stop after any image and resume where you left off.

Run `%gui qt` in a cell first if the viewer doesn't appear.

In [7]:
"""Step 1 — flatten the .ome.tiff tiles into data_tiff/ and build the manifest."""
import json
import re
import shutil
from pathlib import Path

import tifffile

DATA_DIR = Path('data')
TIFF_DIR = Path('data_tiff')          # flat copies live here
RAW_MASK_DIR = Path('masks_raw')      # micro-sam output, before curation
CUR_MASK_DIR = Path('masks_curated')  # what the measurements are taken from
for d in (TIFF_DIR, RAW_MASK_DIR, CUR_MASK_DIR):
    d.mkdir(exist_ok=True)

# EDF_sigma-55_VB-D8FT-0007  ->  sigma 55, constructor VB, condition D8FT, replicate 0007
NAME_RE = re.compile(
    r'^EDF_sigma-(?P<sigma>\d+)_(?P<constructor>[A-Za-z]+)-(?P<condition>D\d+[A-Za-z]*)-(?P<replicate>\d+)$'
)

PHYS_RE = re.compile(r'PhysicalSizeX="([\d.]+)"')
UNIT_RE = re.compile(r'PhysicalSizeXUnit="([^"]*)"')


def pixel_size_um(ome_path):
    """Physical pixel size in µm from the OME header, or None if absent/not µm."""
    with tifffile.TiffFile(ome_path) as tif:
        meta = tif.ome_metadata
    if not meta:
        return None
    size, unit = PHYS_RE.search(meta), UNIT_RE.search(meta)
    if size is None:
        return None
    # the unit is written as 'µm' but the header encoding makes it awkward to compare exactly
    if unit is not None and 'm' not in unit.group(1):
        print(f'  ! {ome_path.name}: unexpected unit {unit.group(1)!r}, treating value as µm')
    return float(size.group(1))


manifest = []
for ome_path in sorted(DATA_DIR.glob('*/*.ome.tiff')):
    fileset = ome_path.parent.name
    stem = ome_path.name[: -len('.ome.tiff')]
    key = f'{fileset}__{stem}'            # unique, traceable back to the source folder
    tiff_path = TIFF_DIR / f'{key}.tiff'
    if not tiff_path.exists():
        shutil.copy2(ome_path, tiff_path)  # byte-identical, just a name without .ome

    fields = NAME_RE.match(stem)
    if fields is None:
        print(f'  ! could not parse {stem!r} — constructor/condition left blank')
    fields = fields.groupdict() if fields else {}

    manifest.append({
        'key': key,
        'fileset': fileset,
        'file_name': stem,
        'tiff_path': str(tiff_path),
        'constructor': fields.get('constructor'),
        'condition': fields.get('condition'),
        'replicate': fields.get('replicate'),
        'pixel_size_um': pixel_size_um(ome_path),
    })

Path('manifest.json').write_text(json.dumps(manifest, indent=1))

missing_scale = [m['key'] for m in manifest if m['pixel_size_um'] is None]
scales = sorted({m['pixel_size_um'] for m in manifest if m['pixel_size_um']})
counts = {}
for m in manifest:
    counts[m['condition']] = counts.get(m['condition'], 0) + 1

print(f'{len(manifest)} tiles -> {TIFF_DIR}/')
print(f'conditions: {counts}')
print(f'pixel size(s) found: {scales} µm/px' + (f'  ({len(missing_scale)} missing!)' if missing_scale else ''))

# The earlier run also left .tiff copies inside data/Fileset_*/. They are redundant now.
# Uncomment to remove them (deletes files — read it before running):
# for p in DATA_DIR.glob('*/*.tiff'):
#     if not p.name.endswith('.ome.tiff'):
#         p.unlink()

91 tiles -> data_tiff/
conditions: {'D8C': 27, 'D2': 16, 'D5': 22, 'D8FT': 26}
pixel size(s) found: [0.52, 0.690625] µm/px


In [8]:
"""Step 2 — automatic segmentation with micro-sam (vit_b_lm + AIS), cached to masks_raw/."""
import time

import numpy as np
import tifffile
import torch
from micro_sam.automatic_segmentation import (
    automatic_instance_segmentation,
    get_predictor_and_segmenter,
)

MODEL_TYPE = 'vit_b_lm'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}' + (f' ({torch.cuda.get_device_name(0)})' if device == 'cuda' else ''))

todo = [m for m in manifest if not (RAW_MASK_DIR / f'{m["key"]}_labels.tiff').exists()]
print(f'{len(manifest) - len(todo)} already segmented, {len(todo)} to do.\n')

if todo:
    predictor, segmenter = get_predictor_and_segmenter(
        model_type=MODEL_TYPE, device=device, segmentation_mode='ais',
    )
    t0 = time.time()
    for i, m in enumerate(todo, 1):
        image = tifffile.imread(m['tiff_path'])
        labels = automatic_instance_segmentation(
            predictor, segmenter, input_path=image, ndim=2, verbose=False,
        )
        tifffile.imwrite(
            RAW_MASK_DIR / f'{m["key"]}_labels.tiff',
            labels.astype(np.uint16), compression='zlib',
        )
        print(f'[{i:2d}/{len(todo)}] {m["file_name"]:<28} {labels.max():3d} objects  '
              f'({time.time() - t0:.0f}s elapsed)')
    print(f'\nSegmented {len(todo)} images in {time.time() - t0:.0f}s.')
else:
    print('Nothing to do — delete files from masks_raw/ to re-segment.')

device: cuda (NVIDIA GeForce RTX 5060 Ti)
0 already segmented, 91 to do.

[ 1/91] EDF_sigma-50_VB-D8C-0001      53 objects  (0s elapsed)
[ 2/91] EDF_sigma-50_VB-D8C-0006      28 objects  (1s elapsed)
[ 3/91] EDF_sigma-50_KT-D8C-0001      83 objects  (1s elapsed)
[ 4/91] EDF_sigma-50_VB-D2-0003       67 objects  (2s elapsed)
[ 5/91] EDF_sigma-50_KT-D2-0005       20 objects  (2s elapsed)
[ 6/91] EDF_sigma-50_KT-D8C-0005      71 objects  (2s elapsed)
[ 7/91] EDF_sigma-50_KT-D5-0005       39 objects  (3s elapsed)
[ 8/91] EDF_sigma-50_VB-D5-0004       71 objects  (3s elapsed)
[ 9/91] EDF_sigma-50_VB-D2-0002       28 objects  (4s elapsed)
[10/91] EDF_sigma-50_VB-D5-0002       92 objects  (4s elapsed)
[11/91] EDF_sigma-50_KT-D8C-0002      43 objects  (4s elapsed)
[12/91] EDF_sigma-50_KT-D2-0003       47 objects  (5s elapsed)
[13/91] EDF_sigma-50_VB-D8C-0002      46 objects  (5s elapsed)
[14/91] EDF_sigma-50_KT-D8C-0006      86 objects  (5s elapsed)
[15/91] EDF_sigma-50_VB-D5-0003      112 obj

In [14]:
"""Step 3 — curate one image at a time.  left-click = delete · u = undo · n = next · x = skip image."""
import json

import napari
import numpy as np
import tifffile
from skimage.segmentation import clear_border, relabel_sequential

SKIP_LOG = Path('skipped_images.json')
skipped = json.loads(SKIP_LOG.read_text()) if SKIP_LOG.exists() else {}


def curated_path(key):
    return CUR_MASK_DIR / f'{key}_labels.tiff'


# only images that are neither curated nor already skipped — this is what makes the cell resumable
queue = [m for m in manifest if not curated_path(m['key']).exists() and m['key'] not in skipped]
print(f'{len(manifest) - len(queue)} of {len(manifest)} already handled, {len(queue)} left to curate.')

state = {'i': 0, 'undo': []}
viewer = napari.Viewer(title='organoid curation')
viewer.text_overlay.visible = True
viewer.text_overlay.font_size = 12

blank = np.zeros((1, 1), np.uint16)
image_layer = viewer.add_image(blank, name='image', colormap='gray')
labels_layer = viewer.add_labels(blank.astype(np.uint32), name='organoids', opacity=0.45)


def _status(note=''):
    m = queue[state['i']]
    n = int(labels_layer.data.max())
    viewer.text_overlay.text = (
        f'[{state["i"] + 1}/{len(queue)}]  {m["file_name"]}   ({m["condition"]})\n'
        f'{n} objects   |   click=delete  u=undo  n=next  x=skip\n{note}'
    )


def _load(i):
    m = queue[i]
    image = tifffile.imread(m['tiff_path'])
    raw = tifffile.imread(RAW_MASK_DIR / f'{m["key"]}_labels.tiff')

    # objects clipped by the field edge have meaningless area/perimeter
    trimmed = clear_border(raw)
    n_dropped = len(np.unique(raw)) - len(np.unique(trimmed))
    labels, _, _ = relabel_sequential(trimmed)

    image_layer.data = image
    image_layer.contrast_limits = [float(image.min()), float(image.max())]
    labels_layer.data = labels.astype(np.uint32)
    state['undo'].clear()
    viewer.reset_view()
    _status(f'{n_dropped} border objects auto-removed' if n_dropped else '')


@labels_layer.mouse_drag_callbacks.append
def _delete_on_click(layer, event):
    """Left-click deletes the object under the cursor; a drag still pans the canvas."""
    origin = np.asarray(event.pos, float)
    dragged = False
    yield
    while event.type == 'mouse_move':
        dragged |= bool(np.linalg.norm(np.asarray(event.pos, float) - origin) > 5)
        yield
    if dragged or event.button != 1:
        return
    value = layer.get_value(
        event.position,
        view_direction=event.view_direction,
        dims_displayed=event.dims_displayed,
        world=True,
    )
    if not value:  # background or outside the image
        return
    where = np.nonzero(layer.data == value)
    state['undo'].append((int(value), where))
    data = layer.data
    data[where] = 0
    layer.data = data
    _status(f'deleted object {int(value)}')


@viewer.bind_key('u', overwrite=True)
def _undo(_viewer):
    if not state['undo']:
        _status('nothing to undo')
        return
    value, where = state['undo'].pop()
    data = labels_layer.data
    data[where] = value
    labels_layer.data = data
    _status(f'restored object {value}')


def _advance(skip_reason=None):
    m = queue[state['i']]
    if skip_reason is None:
        final, _, _ = relabel_sequential(labels_layer.data)
        tifffile.imwrite(curated_path(m['key']), final.astype(np.uint16), compression='zlib')
        print(f'[{state["i"] + 1}/{len(queue)}] {m["file_name"]:<28} kept {int(final.max()):3d} objects')
    else:
        skipped[m['key']] = skip_reason
        SKIP_LOG.write_text(json.dumps(skipped, indent=1))
        print(f'[{state["i"] + 1}/{len(queue)}] {m["file_name"]:<28} SKIPPED ({skip_reason})')

    state['i'] += 1
    if state['i'] >= len(queue):
        viewer.text_overlay.text = 'All images curated — run the measurement cell.'
        print(f'\nDone: {len(queue)} images curated, {len(skipped)} skipped in total.')
        return
    _load(state['i'])


@viewer.bind_key('n', overwrite=True)
def _next(_viewer):
    _advance()


@viewer.bind_key('x', overwrite=True)
def _skip(_viewer):
    _advance(skip_reason='rejected during curation')


if queue:
    _load(0)
else:
    viewer.text_overlay.text = 'Nothing left to curate.'
    print('Nothing left to curate — delete files from masks_curated/ (and entries from '
          'skipped_images.json) to redo any.')

0 of 91 already handled, 91 left to curate.
[1/91] EDF_sigma-50_VB-D8C-0001     kept  46 objects
[2/91] EDF_sigma-50_VB-D8C-0006     kept  27 objects
[3/91] EDF_sigma-50_KT-D8C-0001     kept  69 objects
[4/91] EDF_sigma-50_VB-D2-0003      kept  60 objects
[5/91] EDF_sigma-50_KT-D2-0005      kept  17 objects
[6/91] EDF_sigma-50_KT-D8C-0005     kept  64 objects
[7/91] EDF_sigma-50_KT-D5-0005      kept  37 objects
[8/91] EDF_sigma-50_VB-D5-0004      kept  58 objects
[9/91] EDF_sigma-50_VB-D2-0002      kept  22 objects
[10/91] EDF_sigma-50_VB-D5-0002      kept  77 objects
[11/91] EDF_sigma-50_KT-D8C-0002     kept  29 objects
[12/91] EDF_sigma-50_KT-D2-0003      kept  41 objects
[13/91] EDF_sigma-50_VB-D8C-0002     kept  37 objects


In [12]:
"""Step 4 — measure the curated masks and write the Excel workbook."""
import json

import pandas as pd
import tifffile
from skimage.measure import regionprops_table

XLSX_PATH = Path('organoid_morphometrics.xlsx')
skipped = json.loads(SKIP_LOG.read_text()) if SKIP_LOG.exists() else {}

rows = []
for m in manifest:
    mask_path = CUR_MASK_DIR / f'{m["key"]}_labels.tiff'
    if not mask_path.exists():
        continue  # not curated yet, or skipped
    labels = tifffile.imread(mask_path)
    if labels.max() == 0:
        continue
    um = m['pixel_size_um']  # per file — the tiles were not all acquired at the same scale
    props = pd.DataFrame(regionprops_table(labels, properties=('label', 'area', 'perimeter')))
    props = props.rename(columns={'label': 'object_id', 'area': 'area_px', 'perimeter': 'perimeter_px'})
    props = props[props['perimeter_px'] > 0].copy()
    props['area_um2'] = props['area_px'] * um ** 2
    props['perimeter_um'] = props['perimeter_px'] * um
    props['hydraulic_diameter_um'] = 4 * props['area_um2'] / props['perimeter_um']
    for field in ('file_name', 'fileset', 'constructor', 'condition', 'replicate'):
        props[field] = m[field]
    props['pixel_size_um'] = um
    rows.append(props)

objects = pd.concat(rows, ignore_index=True)[[
    'file_name', 'fileset', 'constructor', 'condition', 'replicate', 'object_id',
    'area_um2', 'perimeter_um', 'hydraulic_diameter_um',
    'area_px', 'perimeter_px', 'pixel_size_um',
]]

METRICS = ['area_um2', 'perimeter_um', 'hydraulic_diameter_um']


def summarise(df, by):
    out = df.groupby(by, as_index=False).agg(
        n_objects=('object_id', 'count'),
        **{f'{metric}_{stat}': (metric, stat)
           for metric in METRICS for stat in ('mean', 'median', 'std')},
    )
    return out.round(3)


by_file = summarise(objects, ['file_name', 'constructor', 'condition', 'replicate'])
by_condition = summarise(objects, ['constructor', 'condition'])
by_condition.insert(2, 'n_images', by_condition['condition'].map(by_file['condition'].value_counts()))

# every input tile accounted for: measured, skipped, or not yet curated
measured = set(objects['file_name'])
status = pd.DataFrame([{
    'file_name': m['file_name'],
    'condition': m['condition'],
    'pixel_size_um': m['pixel_size_um'],
    'status': ('measured' if m['file_name'] in measured
               else 'skipped' if m['key'] in skipped
               else 'not curated'),
    'note': skipped.get(m['key'], ''),
} for m in manifest])

with pd.ExcelWriter(XLSX_PATH, engine='openpyxl') as writer:
    objects.round(3).to_excel(writer, sheet_name='objects', index=False)
    by_file.to_excel(writer, sheet_name='summary_by_file', index=False)
    by_condition.to_excel(writer, sheet_name='summary_by_condition', index=False)
    status.to_excel(writer, sheet_name='image_status', index=False)

print(f'Wrote {XLSX_PATH}')
print(f'{len(objects)} objects from {objects["file_name"].nunique()} images '
      f'({status["status"].value_counts().to_dict()})\n')
print(by_condition.to_string(index=False))

ValueError: No objects to concatenate